In [1]:
import sys, re, time; sys.path.insert(0, "..")
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from src.llm import get_llm
from src.rewrite import retrieve_with_rewrite
from src.retrieval import label

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 20:37:40.940000 60308 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14639.93it/s]


In [2]:
CUTOFF = 0.10
REFUSAL = ("I couldn't find this in the UAE laws I cover (Labour Law, its Executive Regulation, "
           "the 2022 amendment and the PDPL). Please check with MOHRE or a legal professional.")

ANSWER_SYSTEM = """You answer questions about UAE labour law and personal data protection.
Use ONLY the numbered sources below. Do not use outside knowledge.

Rules:
1. End every sentence that states a fact with its source tag, e.g. [S1] or [S1][S3].
2. If the sources do not answer the question, reply with exactly: NOT_FOUND
3. If the Executive Regulation adds detail to a Labour Law article, include that detail.
4. If the 2022 Amendment changes an article, follow the amended text and say it was amended.
5. Be short and clear: 2-6 sentences or a short list. Copy numbers exactly (days, %, periods).
6. Explain what the law says; do not give personal legal advice.
7. If the user asks what an article says, summarise that article from the sources.
8. If the user names an article number without naming the law, use the law in the sources
   that contains that article, and name the law in your answer.

Sources:
{context}"""

answer_chain = (ChatPromptTemplate.from_messages([("system", ANSWER_SYSTEM), ("human", "{question}")])
                | get_llm() | StrOutputParser())

def cite(d):
    m = d.metadata
    return f"{m['title']}, " + ("Preamble" if m["article"] == 0 else f"Article {m['article']}")

def format_context(results):
    return "\n\n".join(f"[S{i}] {cite(d)}\n{d.page_content}" for i, (d, _) in enumerate(results, 1))

In [3]:
from langsmith import traceable

@traceable(name="answer_question")        # groups rewrite + retrieval + answer under ONE trace
def answer(question, top_n=4):
    t0 = time.time()
    q2, results = retrieve_with_rewrite(question, top_n=top_n)        # rewrite -> hybrid -> rerank
    top = results[0][1] if results else 0.0
    out = {"question": question, "rewritten": q2, "top_score": round(top, 3)}

    if top < CUTOFF:                                                   # refusal 1: nothing relevant
        return {**out, "answer": REFUSAL, "sources": [], "path": "refused_low_score",
                "seconds": round(time.time() - t0, 1)}

    results = [(d, s) for d, s in results if s >= CUTOFF]              # drop weak chunks
    text = answer_chain.invoke({"context": format_context(results), "question": question}).strip()

    if text.startswith("NOT_FOUND"):                                   # refusal 2: LLM says not covered
        return {**out, "answer": REFUSAL, "sources": [], "path": "refused_by_llm",
                "seconds": round(time.time() - t0, 1)}

    used = sorted({int(n) for n in re.findall(r"\[S(\d+)\]", text)})   # which sources were cited
    sources = [cite(results[i - 1][0]) for i in used if 0 < i <= len(results)]
    return {**out, "answer": text, "sources": sources,
            "path": "answered" if used else "answered_no_citation",
            "seconds": round(time.time() - t0, 1)}

In [4]:
tests = [
    "How many days of annual leave do I get?",
    "What does Article 51 say?",
    "Can my employer keep my passport?",
    "Can my company share my personal data with others?",
    "What did the 2022 amendment change in the Labour Law?",
    "What is the fee for a UAE Golden Visa?",            # should be refused
]
for q in tests:
    r = answer(q)
    print(f"\n{'=' * 90}\nQ: {q}\n   path={r['path']}  top_score={r['top_score']}  time={r['seconds']}s")
    print(r["answer"])
    print("   Sources:", r["sources"])


Q: How many days of annual leave do I get?
   path=answered  top_score=0.999  time=2.2s
You are entitled to annual leave based on your service period:

- 30 days per year if you have completed one year or more of continuous service [S1].
- 2 days per month if your service is more than six months but less than one year [S1].
- A proportional leave for the part of the year you worked, if your employment ends before using your leave balance [S1].

For part-time workers, the leave is calculated based on actual working hours, with a minimum of 5 working days per year [S2].
   Sources: ['UAE Labour Law (Federal Decree-Law No. 33 of 2021), Article 29', 'Executive Regulation (Cabinet Resolution No. 1 of 2022), Article 18']

Q: What does Article 51 say?
   path=answered  top_score=0.975  time=2.7s
Article 51 of the UAE Labour Law (Federal Decree-Law No. 33 of 2021) outlines the end-of-service gratuity for full-time workers.  
- Foreign full-time workers with one or more years of continuous ser

In [4]:
q = "What does Article 51 say?"
r = answer(q)
print("Rewritten:", r["rewritten"], "\n")
print(r["answer"], "\n")
_, res = retrieve_with_rewrite(q, rewritten=r["rewritten"], top_n=4)   # same sources the model saw
for i, (d, s) in enumerate(res, 1):
    print(f"[S{i}] {s:.3f}  {label(d)}")

Rewritten: What does Article 51 of the UAE Labour Law state regarding the termination of employment? 

Article 51 of the UAE Labour Law (Federal Decree-Law No. 33 of 2021) outlines the end-of-service gratuity for full-time workers. It states that foreign full-time workers who have completed one or more years of continuous service are entitled to an end-of-service gratuity calculated based on their basic wage: 21 days' wage for each year of the first five years, and 30 days' wage for each year beyond that [S3]. The gratuity is calculated using the last basic wage received, or the average daily wage for piece-rate workers, and must not exceed two years' wage [S3][S4]. Days of absence without pay are not included in the service calculation [S3]. Employers may deduct lawful or court-ordered amounts from the gratuity, as per the Executive Regulation [S4]. The Cabinet may approve alternative gratuity systems with specific conditions [S4]. 

[S1] 0.975  labour_law:42
[S2] 0.972  labour_law:47